# 01 — Telecom Week 1: contract, translation, and locked truth

**Outcome:** turn the native telecom fixture into two physically separate
products:

- `SPEC-CORE`: everything a detector may read.
- `SPEC-EVAL`: labels and truth used only after detector output is frozen.

The contract and Telecom Pack are visible below. Only the already-verified,
easy-to-get-silently-wrong mechanics are imported from `week1_core.py`.

This notebook also runs the primary leakage proof: translate an original
source containing truth and a redacted source without truth, then compare
canonical `SPEC-CORE` content hashes.

## 1. Setup

Put this notebook and `week1_core.py` together in:

`MyDrive/anomaly_detection/research/week1/`

In Colab, the next cell mounts Drive. In local Jupyter, set
`ANOMALY_DRIVE_ROOT` and `ANOMALY_NOTEBOOK_HOME` environment variables.

In [ ]:
import ast
import json
import math
import os
import shutil
import sys
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_ROOT = Path(os.getenv(
    "ANOMALY_DRIVE_ROOT",
    "/content/drive/MyDrive/anomaly_detection",
))
NOTEBOOK_HOME = Path(os.getenv(
    "ANOMALY_NOTEBOOK_HOME",
    DRIVE_ROOT / "research" / "week1",
))
if str(NOTEBOOK_HOME) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_HOME))

from week1_core import (
    CORE_TABLES,
    CORE_VERSION,
    EVAL_TABLES,
    FEC_CEILING,
    Selection,
    audit_telecom_output,
    discover_telecom,
    materialise_telecom,
    native_path,
    read_core_bundle,
    runtime_probe,
    timestamp_canary_leaks,
    write_json,
)

print("Contract version:", CORE_VERSION)
print("Drive root:", DRIVE_ROOT)

## 2. Run controls

Every run has a new ID. The notebook refuses to overwrite an existing run,
so an old result always remains traceable to its contract version.

For a short development run, set `ENTITY_IDS` or a time window. For the
final fixture, leave them empty. `BATCH_NATIVE_ROWS` controls physical
memory use; it does not change logical results.

In [ ]:
default_source = DRIVE_ROOT / "Full dataset"
TELECOM_SOURCE = Path(os.getenv(
    "TELECOM_SOURCE_ROOT",
    str(default_source if default_source.exists() else DRIVE_ROOT),
))
RUN_ID = os.getenv("TELECOM_RUN_ID", "telecom_full_v1")
RUN_ROOT = (
    DRIVE_ROOT / "outputs" / "research" / f"v{CORE_VERSION}"
    / "telecom" / RUN_ID
)

SAMPLE_START = os.getenv("TELECOM_SAMPLE_START") or None
SAMPLE_END = os.getenv("TELECOM_SAMPLE_END") or None
ENTITY_IDS = tuple(filter(
    None,
    (item.strip() for item in os.getenv("TELECOM_ENTITY_IDS", "").split(",")),
))
BATCH_NATIVE_ROWS = int(os.getenv("TELECOM_BATCH_NATIVE_ROWS", "100000"))
MEMORY_BUDGET_GIB = float(os.getenv("TELECOM_MEMORY_BUDGET_GIB", "8"))
RUN_MATERIALISATION = os.getenv("RUN_MATERIALISATION", "1") == "1"
RUN_ACCEPTANCE = os.getenv("RUN_ACCEPTANCE", "1") == "1"

selection = Selection(
    sample_start=SAMPLE_START,
    sample_end=SAMPLE_END,
    entity_ids=ENTITY_IDS,
    batch_native_rows=BATCH_NATIVE_ROWS,
)
display(pd.Series({
    "source": str(TELECOM_SOURCE),
    "run_root": str(RUN_ROOT),
    "sample_start": SAMPLE_START,
    "sample_end": SAMPLE_END,
    "entity_ids": ENTITY_IDS or "all",
    "batch_native_rows": BATCH_NATIVE_ROWS,
}, name="value").to_frame())

## 3. Neutral contract — visible and sector-independent

`SPEC-CORE` contains no labels, fault IDs, true intervals, ticket outcome
labels, `class`, or `state`. Entity validity is stored once in
`entity_registry.valid_from/valid_to`; there is deliberately no duplicate
canonical `entity_service_windows` table.

Three situations stay distinct:

1. a telemetry row is present with a null value (`quality_code=invalid`);
2. an observation was expected but absent (`collection_gaps`);
3. an entity was outside its validity window (not a collection gap).

`SPEC-EVAL` is physically separable and may be removed before scoring.
`gt_condition_states` is included because Petrobras 3W supplies labelled
states as well as events. It intentionally has no untested
`severity_ordinal`.

In [ ]:
SPEC_CORE = {
    "telemetry": [
        "event_ts", "ingested_at", "entity_id", "metric_id", "value",
        "quality_code", "quality_detail", "exposure", "source_id",
    ],
    "metric_catalogue": [
        "metric_id", "entity_type", "measurement_kind", "unit",
        "anomaly_direction", "sampling_semantics",
        "aggregation_semantics", "value_nullable",
        "expected_cadence_seconds", "censoring_type",
        "expected_behaviour_profile", "lower_bound", "upper_bound",
        "candidate_periods", "exposure_metric_id",
        "exposure_semantics", "exposure_unit",
        "exposure_formula_id", "exposure_source", "context_keys",
    ],
    "entity_registry": [
        "entity_id", "entity_type", "valid_from", "valid_to",
        "attributes_json", "source_id",
    ],
    "entity_relations": [
        "parent_entity_id", "child_entity_id", "relation_type",
        "relation_family", "valid_from", "valid_to",
        "relation_confidence", "source",
    ],
    "operational_events": [
        "event_id", "entity_id", "event_type", "event_start",
        "event_end", "known_at", "attributes_json", "source",
    ],
    "collection_gaps": [
        "entity_id", "gap_start", "gap_end", "known_at", "source",
    ],
}
SPEC_EVAL = {
    "gt_fault_events": [
        "fault_event_id", "fault_type", "fault_family",
        "fault_domain_type", "fault_domain_id", "onset_ts",
        "first_observable_ts", "impact_ts", "resolution_ts",
        "cause_group_id", "left_censored", "label_source",
        "source_instance_id",
    ],
    "gt_fault_entity_intervals": [
        "fault_event_id", "affected_entity_id", "fault_family",
        "channel", "active_start_ts", "active_end_ts", "impact_ts",
        "contribution",
    ],
    "gt_cause_groups": [
        "cause_group_id", "cause_type", "start_ts", "end_ts",
        "footprint_json",
    ],
    "gt_condition_states": [
        "entity_id", "condition_start_ts", "condition_end_ts",
        "condition_code", "condition_label", "label_source",
        "source_instance_id",
    ],
    "gt_benign_anomalies": [
        "entity_id", "event_ts", "benign_type", "n_samples",
    ],
    "gt_collection_gaps": [
        "entity_id", "event_ts", "gap_reason",
    ],
    "gt_ticket_links": [
        "ticket_id", "entity_id", "fault_event_id",
        "fault_type_label", "is_no_fault_found", "is_misattributed",
        "reported_ts", "resolved_ts",
    ],
}

assert tuple(SPEC_CORE) == CORE_TABLES
assert set(SPEC_EVAL).issubset(EVAL_TABLES)
display(pd.DataFrame({
    "SPEC-CORE table": list(SPEC_CORE),
    "purpose": [
        "measurements", "metric meaning", "entity identity and validity",
        "topology plus non-tree relations", "known operational context",
        "expected observations that were absent",
    ],
}))
display(pd.DataFrame({"SPEC-EVAL table": list(SPEC_EVAL)}))

## 4. Telecom Pack — the telco phrasebook

The pack maps native GPON names into the neutral vocabulary. It is not the
generic contract. A new sector supplies another mapping table without
adding telecom branches to the core translator.

`anomaly_direction` uses only: `decrease`, `increase`, `both`, `change`.
Candidate periods are hypotheses, not facts.

In [ ]:
def metric(
    native_field, metric_id, kind, unit, direction, behaviour,
    *, sampling, aggregation, nullable=True, censoring="none",
    lower=None, upper=None, periods=(), exposure_semantics=None,
    exposure_unit=None, exposure_formula_id=None,
    exposure_source=None, context=(),
):
    return {
        "native_field": native_field,
        "metric_id": metric_id,
        "entity_type": "ont",
        "measurement_kind": kind,
        "unit": unit,
        "anomaly_direction": direction,
        "sampling_semantics": sampling,
        "aggregation_semantics": aggregation,
        "value_nullable": nullable,
        "expected_cadence_seconds": 900,
        "censoring_type": censoring,
        "expected_behaviour_profile": behaviour,
        "lower_bound": lower,
        "upper_bound": upper,
        "candidate_periods": list(periods),
        "exposure_metric_id": None,
        "exposure_semantics": exposure_semantics,
        "exposure_unit": exposure_unit,
        "exposure_formula_id": exposure_formula_id,
        "exposure_source": exposure_source,
        "context_keys": list(context),
    }

metric_catalogue = pd.DataFrame([
    metric("rx_power_dbm", "telecom.optical.rx_power", "gauge", "dBm",
           "decrease", "stable_continuous",
           sampling="instantaneous_at_poll", aggregation="median",
           censoring="quantised", periods=("P1D", "P7D"),
           context=("enclosure", "distance_bucket", "splitter_ratio")),
    metric("tx_power_dbm", "telecom.optical.tx_power", "gauge", "dBm",
           "both", "stable_continuous",
           sampling="instantaneous_at_poll", aggregation="median",
           censoring="quantised", periods=("P1D",),
           context=("device_model", "firmware_version")),
    metric("temperature_c", "telecom.ont.temperature", "gauge", "degC",
           "increase", "trend_one_season",
           sampling="instantaneous_at_poll", aggregation="mean",
           censoring="quantised", periods=("P1D", "P365D"),
           context=("enclosure",)),
    metric("bias_current_ma", "telecom.ont.bias_current", "gauge", "mA",
           "increase", "trend_one_season",
           sampling="instantaneous_at_poll", aggregation="median",
           censoring="quantised", lower=0, periods=("P1D",),
           context=("device_model", "temperature")),
    metric("voltage_v", "telecom.ont.voltage", "gauge", "V",
           "both", "stable_continuous",
           sampling="instantaneous_at_poll", aggregation="median",
           censoring="quantised", lower=0, periods=("P1D",),
           context=("device_model", "throughput")),
    metric("ber", "telecom.link.ber", "bounded_fraction", "ratio",
           "increase", "zero_inflated_bounded",
           sampling="interval_estimate",
           aggregation="exposure_weighted_mean",
           censoring="left_censored_at_reporting_floor",
           lower=0, upper=1, context=("device_model", "temperature")),
    metric("fec_count", "telecom.link.fec_count", "interval_count",
           "count", "increase", "overdispersed_count",
           sampling="generator_defined_corrections_since_previous_poll",
           aggregation="sum",
           censoring="right_censored_at_generator_ceiling",
           lower=0, upper=FEC_CEILING,
           exposure_semantics="transmitted_bit_opportunities_under_frozen_generator",
           exposure_unit="bit_opportunity",
           exposure_formula_id="telemetry_synth_4_0_1_fec_bits",
           exposure_source="frozen_generator_mechanism",
           context=("device_model", "temperature")),
    metric("crc_errors", "telecom.link.crc_errors", "interval_count",
           "count", "increase", "overdispersed_count",
           sampling="generator_defined_crc_errors_since_previous_poll",
           aggregation="sum",
           censoring="generator_cap_precedes_optional_burst", lower=0,
           exposure_semantics="estimated_frame_opportunities_from_observed_throughput",
           exposure_unit="frame_opportunity",
           exposure_formula_id="throughput_mbps_times_interval_over_pack_frame_bits_v1",
           exposure_source="derived_from_native_throughput_with_pack_parameter",
           context=("device_model",)),
    metric("uptime_s", "telecom.ont.uptime", "cumulative_counter", "s",
           "change", "counter_with_resets", sampling="value_at_poll",
           aggregation="last", nullable=False, lower=0),
    metric("reboot_count", "telecom.ont.reboot_count",
           "cumulative_counter", "count", "increase",
           "counter_with_resets", sampling="value_at_poll",
           aggregation="last", nullable=False, lower=0),
    metric("throughput_mbps", "telecom.service.throughput", "gauge",
           "Mbps", "decrease", "multiple_seasonalities",
           sampling="interval_average", aggregation="mean",
           censoring="quantised", lower=0, periods=("P1D", "P7D"),
           context=("service_tier", "day_of_week")),
])

assert set(metric_catalogue["anomaly_direction"]) <= {
    "decrease", "increase", "both", "change"
}
assert metric_catalogue["metric_id"].is_unique
display(metric_catalogue)

### Entity relations and exposure provenance

Geographic membership deliberately creates a second, non-tree relation
for each ONT. It tests that the neutral relation model is not merely a
telecom containment tree.

FEC and CRC exposure are intentionally different:

- FEC uses constant transmitted-bit opportunity under the **frozen
  generator mechanism**.
- CRC derives varying frame opportunities from each row's
  `throughput_mbps`.

The physical standard and generator constants are both recorded. The
translator reports their mismatch; it does not pretend `fec_count` is a
standards-defined corrected-codeword counter.

In [ ]:
relation_mappings = [
    {"parent_field": "olt_id", "child_field": "pon_port",
     "relation_type": "contains_pon",
     "relation_family": "network_topology"},
    {"parent_field": "pon_port", "child_field": "splitter_l1",
     "relation_type": "contains_splitter_l1",
     "relation_family": "network_topology"},
    {"parent_field": "splitter_l1", "child_field": "splitter_l2",
     "relation_type": "contains_splitter_l2",
     "relation_family": "network_topology"},
    {"parent_field": "splitter_l2", "child_field": "ont_id",
     "relation_type": "serves_ont",
     "relation_family": "network_topology"},
    {"parent_field": "geo_cluster", "child_field": "ont_id",
     "relation_type": "groups_ont",
     "relation_family": "geographic_membership"},
]

pack_parameters = {
    "gpon_standard_downstream_line_rate_bps": {
        "value": 2_488_320_000, "unit": "bit/s",
        "provenance": "ITU-T G.984.3 nominal downstream line rate",
        "status": "physical_standard_reference_not_used_to_reinterpret_generator_counts",
    },
    "gpon_standard_rs_255_239_transmitted_bits": {
        "value": 2040, "unit": "bit/codeword",
        "provenance": "RS(255,239): 255 transmitted octets",
        "status": "physical_standard_reference",
    },
    "gpon_standard_rs_255_239_payload_bits": {
        "value": 1912, "unit": "bit/codeword",
        "provenance": "RS(255,239): 239 payload octets",
        "status": "physical_standard_reference",
    },
    "generator_fec_line_rate_bps": {
        "value": 2_488_000_000, "unit": "bit/s",
        "provenance": "telemetry-synth 4.0.1 frozen generator mechanism",
        "status": "canonical_fec_bit_exposure_parameter",
    },
    "generator_fec_divisor_bits": {
        "value": 1904, "unit": "bit",
        "provenance": "telemetry-synth 4.0.1 frozen generator mechanism",
        "status": "lineage_only_not_an_RS_255_239_codeword_size",
    },
    "crc_frame_size_bytes": {
        "value": 1500, "unit": "byte",
        "provenance": "telemetry-synth 4.0.1 faults.py",
        "status": "frozen_source_parameter",
    },
    "generator_crc_reference_rate_bps": {
        "value": 78_000_000, "unit": "bit/s",
        "provenance": "telemetry-synth 4.0.1 faults.py",
        "status": "lineage_only_not_canonical_exposure",
    },
}

display(pd.DataFrame(relation_mappings))
display(pd.DataFrame(pack_parameters).T)

## 5. Discover and translate

The generator adapter reads the native source in bounded batches. Native
`entity_service_windows.csv` remains an input, but canonical validity is
stored only in `entity_registry`. Tickets and all `gt_*` fields route only
to `SPEC-EVAL`.

In [ ]:
inventory = discover_telecom(TELECOM_SOURCE)
display(pd.Series(inventory, name="value").to_frame())
assert inventory["core_ready"], inventory
assert inventory["evaluation_ready"], inventory

if RUN_MATERIALISATION:
    workflow_report = materialise_telecom(
        TELECOM_SOURCE,
        RUN_ROOT,
        metric_catalogue,
        relation_mappings,
        pack_parameters,
        selection=selection,
        include_evaluation=True,
    )
else:
    workflow_report = json.loads(
        (RUN_ROOT / "workflow_report.json").read_text()
    )

display(pd.Series(
    workflow_report["core_manifest"]["row_counts"],
    name="rows",
).to_frame())
display(pd.Series(workflow_report["representation"], name="value").to_frame())

## 6. Materialisation acceptance checks

These assertions are intentionally visible. A green cell proves the
contract shape, quality routing, exposure semantics, geographic edge, and
bounded-memory acceptance rule for this run.

In [ ]:
core = RUN_ROOT / "SPEC-CORE"
evaluation = RUN_ROOT / "SPEC-EVAL"
core_manifest = json.loads((core / "manifest.json").read_text())
audit = audit_telecom_output(core)
relations = pd.read_parquet(core / "entity_relations.parquet")
catalogue_on_disk = pd.read_parquet(core / "metric_catalogue.parquet")
fault_events = pd.read_parquet(evaluation / "gt_fault_events.parquet")
cause_groups = pd.read_parquet(evaluation / "gt_cause_groups.parquet")
lineage = json.loads((core / "translation_lineage.json").read_text())
core_tree = ast.parse((NOTEBOOK_HOME / "week1_core.py").read_text())
local_imports = [
    node.module for node in ast.walk(core_tree)
    if isinstance(node, ast.ImportFrom)
    and node.module
    and node.module.split(".")[0] not in {
        "__future__", "configparser", "gc", "hashlib", "itertools",
        "json", "os", "shutil", "dataclasses", "pathlib", "typing",
        "numpy", "pandas", "pyarrow",
    }
]

assert core.is_dir() and evaluation.is_dir()
assert not (core / "entity_service_windows.parquet").exists()
assert set(CORE_TABLES) == set(core_manifest["row_counts"])
assert not any(column.startswith("gt_") for column in catalogue_on_disk.columns)
assert lineage["tickets_excluded_from_core"] is True
assert lineage["entity_validity_stored_once"] is True
assert local_imports == [], f"unexpected project imports: {local_imports}"
assert "geographic_membership" in set(relations["relation_family"])
referenced_groups = set(fault_events["cause_group_id"].dropna().astype(str))
published_groups = set(cause_groups["cause_group_id"].dropna().astype(str))
assert referenced_groups <= published_groups
assert audit["quality_counts"].get("invalid", 0) > 0
assert audit["quality_counts"].get("clipped", 0) > 0

fec_min, fec_max = audit["exposure_ranges"]["telecom.link.fec_count"]
crc_min, crc_max = audit["exposure_ranges"]["telecom.link.crc_errors"]
assert math.isfinite(fec_min) and fec_min == fec_max
assert math.isfinite(crc_min) and crc_min < crc_max

peak_rss = workflow_report["memory"]["peak_observed_rss_gib"]
if math.isfinite(peak_rss):
    assert peak_rss <= MEMORY_BUDGET_GIB, (peak_rss, MEMORY_BUDGET_GIB)

display(pd.Series(audit["quality_counts"], name="rows").to_frame())
display(pd.DataFrame(audit["exposure_ranges"], index=["min", "max"]).T)
print("PASS — materialisation contract and pack assertions")

## 7. Primary leakage proof: source-tree translator invariance

This is the important lock test because the translator is the component
that can see both worlds.

We create two native fixtures from the same selected records:

- **original:** truth columns, evaluation files, and `tickets.csv` present;
- **redacted:** all truth columns removed and every evaluation file,
  explicitly including `tickets.csv`, absent.

Evaluation timestamps in the original are shifted 50 years to become
unmistakable canaries. Translation must produce identical canonical
`SPEC-CORE` hashes from both sources.

In [ ]:
def build_isolation_sources(source, destination, catalogue):
    original = destination / "original"
    redacted = destination / "redacted"
    original_data = original / "Data"
    redacted_data = redacted / "Data"
    original_eval = original / "evaluation"
    original_data.mkdir(parents=True)
    redacted_data.mkdir(parents=True)
    original_eval.mkdir(parents=True)

    panel_path = native_path(source, "reference_dataset.parquet")
    parquet = pq.ParquetFile(panel_path)
    metric_fields = [
        field for field in catalogue["native_field"]
        if field in parquet.schema_arrow.names
    ]
    scan_fields = [
        field for field in ["fec_count", *metric_fields]
        if field in parquet.schema_arrow.names
    ]
    clipped_entities, invalid_entities = set(), set()
    for batch in parquet.iter_batches(
        batch_size=100_000,
        columns=["ont_id", *scan_fields],
    ):
        frame = batch.to_pandas()
        if "fec_count" in frame:
            clipped_entities.update(
                frame.loc[
                    pd.to_numeric(frame["fec_count"], errors="coerce")
                    .ge(FEC_CEILING),
                    "ont_id",
                ].astype(str).head(1)
            )
        invalid = frame[metric_fields].isna().any(axis=1)
        invalid_entities.update(
            frame.loc[invalid, "ont_id"].astype(str).head(1)
        )
        if clipped_entities and invalid_entities:
            break
    selected_entities = sorted(clipped_entities | invalid_entities)
    if not selected_entities:
        raise AssertionError("could not find clipped/null acceptance records")

    selected_batches = []
    for batch in parquet.iter_batches(batch_size=100_000):
        frame = batch.to_pandas()
        frame = frame.loc[
            frame["ont_id"].astype(str).isin(selected_entities)
        ]
        if len(frame):
            selected_batches.append(frame)
    panel = pd.concat(selected_batches, ignore_index=True)
    panel.to_parquet(original_data / "reference_dataset.parquet", index=False)
    panel.drop(
        columns=[column for column in panel if column.startswith("gt_")],
        errors="ignore",
    ).to_parquet(redacted_data / "reference_dataset.parquet", index=False)

    topology = pd.read_csv(native_path(source, "topology.csv"))
    topology = topology.loc[
        topology["ont_id"].astype(str).isin(selected_entities)
    ]
    topology.to_csv(original_data / "topology.csv", index=False)
    topology.drop(
        columns=[column for column in topology if column.startswith("gt_")],
        errors="ignore",
    ).to_csv(redacted_data / "topology.csv", index=False)

    windows = pd.read_csv(native_path(source, "entity_service_windows.csv"))
    windows = windows.loc[
        windows["entity_id"].astype(str).isin(selected_entities)
    ]
    windows.to_csv(original_data / "entity_service_windows.csv", index=False)
    windows.to_csv(redacted_data / "entity_service_windows.csv", index=False)

    engineering_path = native_path(
        source, "engineering_events.csv", required=False
    )
    if engineering_path:
        engineering = pd.read_csv(engineering_path)
        graph_ids = set(selected_entities)
        for field in [
            "olt_id", "pon_port", "splitter_l1", "splitter_l2", "geo_cluster"
        ]:
            if field in topology:
                graph_ids.update(topology[field].dropna().astype(str))
        engineering = engineering.loc[
            engineering["entity_id"].astype(str).isin(graph_ids)
        ]
        engineering.to_csv(original_data / "engineering_events.csv", index=False)
        engineering.to_csv(redacted_data / "engineering_events.csv", index=False)

    canaries = []
    evaluation_files = {
        "gt_fault_registry.csv": [
            "onset_ts", "first_observable_ts", "impact_ts", "repair_ts"
        ],
        "fault_entity_intervals.csv": [
            "active_start_ts", "active_end_ts", "impact_ts"
        ],
        "gt_fault_groups.csv": ["start_ts", "end_ts"],
        "tickets.csv": ["reported_ts", "resolved_ts"],
        "gt_benign_anomalies.csv": ["ts"],
    }
    for filename, timestamp_fields in evaluation_files.items():
        path = native_path(
            source, filename, evaluation=True, required=False
        )
        if path is None:
            continue
        frame = pd.read_csv(path)
        if "entity_id" in frame:
            frame = frame.loc[
                frame["entity_id"].astype(str).isin(selected_entities)
            ]
        if "ont_id" in frame:
            frame = frame.loc[
                frame["ont_id"].astype(str).isin(selected_entities)
            ]
        for field in timestamp_fields:
            if field in frame:
                shifted = (
                    pd.to_datetime(frame[field], utc=True, errors="coerce")
                    + pd.DateOffset(years=50)
                )
                frame[field] = shifted
                canaries.extend(shifted.dropna().tolist())
        frame.to_csv(original_eval / filename, index=False)

    gap_path = native_path(
        source, "gt_collection_gaps.parquet",
        evaluation=True, required=False,
    )
    if gap_path:
        gaps = pd.read_parquet(gap_path)
        if "entity_id" in gaps:
            gaps = gaps.loc[
                gaps["entity_id"].astype(str).isin(selected_entities)
            ]
        if "ts" in gaps:
            gaps["ts"] = (
                pd.to_datetime(gaps["ts"], utc=True, errors="coerce")
                + pd.DateOffset(years=50)
            )
            canaries.extend(gaps["ts"].dropna().tolist())
        gaps.to_parquet(
            original_eval / "gt_collection_gaps.parquet", index=False
        )
    return original, redacted, selected_entities, canaries

In [ ]:
isolation_result = {"status": "skipped"}
if RUN_ACCEPTANCE:
    with tempfile.TemporaryDirectory(prefix="week1-isolation-") as tmp:
        tmp = Path(tmp)
        original, redacted, test_entities, canaries = build_isolation_sources(
            TELECOM_SOURCE, tmp / "sources", metric_catalogue
        )
        test_selection = Selection(
            entity_ids=tuple(test_entities),
            batch_native_rows=10_000,
        )
        original_run = tmp / "original-run"
        redacted_run = tmp / "redacted-run"
        original_report = materialise_telecom(
            original, original_run, metric_catalogue,
            relation_mappings, pack_parameters,
            selection=test_selection, include_evaluation=True,
        )
        redacted_report = materialise_telecom(
            redacted, redacted_run, metric_catalogue,
            relation_mappings, pack_parameters,
            selection=test_selection, include_evaluation=False,
        )

        original_hashes = original_report["core_manifest"][
            "canonical_content_hashes"
        ]
        redacted_hashes = redacted_report["core_manifest"][
            "canonical_content_hashes"
        ]
        assert original_hashes == redacted_hashes

        original_core = read_core_bundle(original_run / "SPEC-CORE")
        assert not timestamp_canary_leaks(original_core, canaries)
        assert "clipped" in set(original_core["telemetry"]["quality_code"])
        assert "invalid" in set(original_core["telemetry"]["quality_code"])

        # Negative control: a deliberately leaky table must be detected.
        deliberately_leaky = {
            name: frame.copy() for name, frame in original_core.items()
        }
        canary = pd.Timestamp(canaries[0])
        deliberately_leaky["operational_events"] = pd.DataFrame({
            "event_start": [canary],
        })
        detected = timestamp_canary_leaks(deliberately_leaky, [canary])
        assert detected == ["operational_events.event_start"]

        # Secondary deployment evidence only: the placeholder runtime gets
        # a SPEC-CORE path, never a SPEC-EVAL path or argument.
        present = runtime_probe(original_run / "SPEC-CORE")
        (original_run / "SPEC-EVAL").rename(original_run / "EVAL-HIDDEN")
        renamed = runtime_probe(original_run / "SPEC-CORE")
        shutil.rmtree(original_run / "EVAL-HIDDEN")
        removed = runtime_probe(original_run / "SPEC-CORE")
        (original_run / "SPEC-EVAL").mkdir()
        empty = runtime_probe(original_run / "SPEC-CORE")
        assert len({present, renamed, removed, empty}) == 1

        # A detector-time table may not expose future-known events.
        events = original_core["operational_events"].copy()
        if len(events):
            events["event_start"] = pd.to_datetime(events["event_start"], utc=True)
            events["known_at"] = events["event_start"] + pd.Timedelta(days=1)
            as_of = events["event_start"].max() + pd.Timedelta(hours=1)
            visible = events.loc[
                pd.to_datetime(events["known_at"], utc=True) <= as_of
            ]
            assert visible.empty

        isolation_result = {
            "status": "passed",
            "test_entities": test_entities,
            "canonical_hashes_equal": True,
            "timestamp_canaries_checked": len(canaries),
            "negative_control_detected": detected,
            "runtime_variants_equal": True,
            "runtime_test_role": (
                "secondary deployment evidence; translator invariance "
                "is the primary leakage proof"
            ),
        }

display(pd.Series(isolation_result, name="value").to_frame())
print("PASS — translator invariance and negative control")

## 8. Save the acceptance record

The detector/modelling notebook may now consume `SPEC-CORE` with
`SPEC-EVAL` unmounted. Evaluation is allowed only after scores and ranked
incidents have been saved.

In [ ]:
acceptance_report = {
    "contract_version": CORE_VERSION,
    "run_root": str(RUN_ROOT),
    "spec_core_tables": list(SPEC_CORE),
    "spec_eval_tables": list(SPEC_EVAL),
    "quality_counts": audit["quality_counts"],
    "exposure_ranges": audit["exposure_ranges"],
    "entity_validity_stored_once": True,
    "tickets_excluded_from_core": True,
    "geographic_relation_exercised": True,
    "grouped_fault_references_valid": True,
    "dependency_direction": "notebooks import one standalone core file; core imports no project modules",
    "memory_budget_gib": MEMORY_BUDGET_GIB,
    "peak_observed_rss_gib": peak_rss,
    "truth_isolation": isolation_result,
}
report_path = RUN_ROOT / "acceptance_report.json"
if not report_path.exists():
    write_json(report_path, acceptance_report)
print("Saved:", report_path)
print("NEXT: run Notebook 02 and Notebook 03.")